### Case Study Part 1: Environment Setup
    In this step, we import all the essential libraries required for our loan recovery analytics project. This includes libraries for data manipulation (`pandas`, `numpy`), interactive visualizations (`plotly`), and machine learning algorithms (`scikit-learn` for clustering, scaling, and classification).

In [1]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("✅ Saari libraries successfully import ho gayi hain!")


✅ Saari libraries successfully import ho gayi hain!


### Case Study Part 2: Data Loading & Preliminary Exploration
Here, we load the loan recovery dataset from a CSV file. We then print the first 5 rows to understand the structure of the data and perform a statistical summary (`describe`) to analyze the distribution, mean, standard deviation, and range of our numerical variables.

In [2]:
# Data load karna (Apna correct path ensure karein)
df = pd.read_csv("/content/loan-recovery (2).csv")

print("--- Dataset Ke Shuruati 5 Rows ---")
print(df.head())

print("\n--- Data Ki Mathematical Summary ---")
print(df.describe())


--- Dataset Ke Shuruati 5 Rows ---
  Borrower_ID  Age  Gender Employment_Type  Monthly_Income  Num_Dependents  \
0       BRW_1   59    Male        Salaried          215422               0   
1       BRW_2   49  Female        Salaried           60893               0   
2       BRW_3   35    Male        Salaried          116520               1   
3       BRW_4   63  Female        Salaried          140818               2   
4       BRW_5   28    Male        Salaried           76272               1   

  Loan_ID  Loan_Amount  Loan_Tenure  Interest_Rate  ... Collateral_Value  \
0    LN_1      1445796           60          12.39  ...     1.727997e+06   
1    LN_2      1044620           12          13.47  ...     1.180032e+06   
2    LN_3      1923410           72           7.74  ...     2.622540e+06   
3    LN_4      1811663           36          12.23  ...     1.145493e+06   
4    LN_5        88578           48          16.13  ...     0.000000e+00   

   Outstanding_Loan_Amount  Monthly_EMI

### Case Study Part 3: Loan Amount Distribution Analysis
To understand how loan amounts are distributed among borrowers, we create an interactive histogram using Plotly. This visualization also includes a violin plot on the margin, helping us identify major peaks and the overall spread of loan disbursements.

In [3]:
fig1 = px.histogram(df, x='Loan_Amount', nbins=30, marginal="violin", opacity=0.7,
                   title="Loan Amount Distribution",
                   labels={'Loan_Amount': "Loan Amount (in $)"},
                   color_discrete_sequence=["royalblue"])
fig1.update_layout(template="plotly_white")
fig1.show()


### Case Study Part 4: Relationship Between Loan Amount & Monthly Income
This scatter plot helps us understand if borrower income dictates the size of the loan they receive. We use size and color encodings to map the loan amount, showing a clear trend where high-income individuals generally receive larger loans.

In [4]:
fig2 = px.scatter(df, x='Loan_Amount', y='Monthly_Income',
                     color='Loan_Amount', color_continuous_scale='Viridis',
                     size='Loan_Amount', hover_name=df.index,
                     title="Relationship between Loan Amount & Monthly Income")

fig2.update_layout(
    annotations=[
        dict(
            x=df['Loan_Amount'].max() * 0.8, y=df['Monthly_Income'].max(),
            text="Higher Loan Amounts are linked to Higher Income Levels",
            showarrow=True,
            arrowhead=2,
            font=dict(size=12, color="red")
        )
    ],
    xaxis_title="Loan Amount (in $)",
    yaxis_title="Monthly Income (in $)",
    template="plotly_white"
)
fig2.show()


### Case Study Part 5: Impact of Payment History on Recovery Status
We analyze the categorical relationship between the past payment histories of borrowers and their final recovery outcomes. This stacked histogram highlights how timely or delayed historic payments influence the likelihood of successful loan recovery.

In [5]:
fig3 = px.histogram(df, x="Payment_History", color="Recovery_Status", barmode="group",
                   title="How Payment History Affects Loan Recovery Status",
                   labels={"Payment_History": "Payment History", "count": "Number of Loans"},
                   color_discrete_map={"Recovered": "green", "Not Recovered": "red"})

fig3.update_layout(
    xaxis_title="Payment History",
    yaxis_title="Number of Loans",
    legend_title="Recovery Status",
    template="plotly_white"
)
fig3.show()


### Case Study Part 6: Analysis of Missed Payments vs. Recovery
This box plot visualizes how the number of missed payments correlates with the recovery status. It helps us establish a threshold of missed payments beyond which a loan becomes extremely difficult to recover.

In [6]:
fig4 = px.box(df, x="Recovery_Status", y="Num_Missed_Payments",
             title="How Missed Payments Affect Loan Recovery Status",
             labels={"Recovery_Status": "Recovery Status", "Num_Missed_Payments": "Number of Missed Payments"},
             color="Recovery_Status",
             color_discrete_map={"Recovered": "green", "Not Recovered": "red"},
             points="all")

fig4.update_layout(
    xaxis_title="Recovery Status",
    yaxis_title="Number of Missed Payments",
    template="plotly_white"
)
fig4.show()


### Case Study Part 7: Unsupervised Clustering for Borrower Segmentation
Using K-Means clustering, we group borrowers into 4 distinct segments based on normalized features. Standardizing the data ensures that features with larger ranges do not disproportionately impact distance calculations.

In [7]:
# 1. Feature list ready karna
features = ['Age', 'Monthly_Income', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate',
            'Collateral_Value', 'Outstanding_Loan_Amount', 'Monthly_EMI', 'Num_Missed_Payments', 'Days_Past_Due']

# 2. Data Standardize karna distance sahi calculate karne ke liye
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df[features])

# 3. K-Means apply karna clusters banane ke liye
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=15, max_iter=500)
df['Borrower_Segment'] = kmeans.fit_predict(df_scaled)

print("🎯 Unsupervised Clustering Complete! 'Borrower_Segment' column add ho gaya hai.")


🎯 Unsupervised Clustering Complete! 'Borrower_Segment' column add ho gaya hai.


### Case Study Part 8: Visualizing Borrower Segments (2D)
We construct a 2D scatter plot using Plotly to visually verify the segments created by K-Means, mapping Monthly Income against Loan Amount to identify clear clusters.

In [8]:
fig6 = px.scatter(df, x='Monthly_Income', y='Loan_Amount',
                 color=df['Borrower_Segment'].astype(str), size='Loan_Amount',
                 hover_data={'Monthly_Income': True, 'Loan_Amount': True, 'Borrower_Segment': True},
                 title="Borrower Segments Based on Monthly Income and Loan Amount",
                 labels={"Monthly_Income": "Monthly Income ($)", "Loan_Amount": "Loan Amount ($)", "Borrower_Segment": "Segment"},
                 color_discrete_sequence=px.colors.qualitative.Vivid)

fig6.update_layout(
    xaxis_title="Monthly Income ($)",
    yaxis_title="Loan Amount ($)",
    template="plotly_white",
    legend_title="Borrower Segment"
)
fig6.show()


### Case Study Part 9: 3D Visualization of Borrower Clusters
To gain a deeper perspective, we project the clusters into a 3D space by adding a third dimension: the number of missed payments. This visualization helps identify high-risk borrower clusters.

In [9]:
import plotly.express as px

fig_3d_segments = px.scatter_3d(
    df,
    x='Monthly_Income',
    y='Loan_Amount',
    z='Num_Missed_Payments',
    color=df['Borrower_Segment'].astype(str),  # 4 Clusters ko alag color dene ke liye
    title='1. 3D Borrower Segments (Income vs Loan vs Missed Payments)',
    labels={
        'Monthly_Income': 'Monthly Income ($)',
        'Loan_Amount': 'Loan Amount ($)',
        'Num_Missed_Payments': 'Missed Payments'
    },
    color_discrete_sequence=px.colors.qualitative.Vivid,
    opacity=0.8
)

# Graph layout clean karna
fig_3d_segments.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    legend_title="Borrower Segment"
)
fig_3d_segments.show()


### Case Study Part 10: 3D Visualization of Recovery Status
In this step, we keep the same 3D coordinates (Income, Loan, Missed Payments) but change the color mapping to reflect the actual Recovery Status. This allows us to compare unsupervised cluster profiles with real historical outcomes.

In [10]:
import plotly.express as px

fig_3d_recovery = px.scatter_3d(
    df,
    x='Monthly_Income',
    y='Loan_Amount',
    z='Num_Missed_Payments',
    color='Recovery_Status',  # Recovered aur Not Recovered ko alag karne ke liye
    title='2. 3D Loan Recovery Status (Income vs Loan vs Missed Payments)',
    labels={
        'Monthly_Income': 'Monthly Income ($)',
        'Loan_Amount': 'Loan Amount ($)',
        'Num_Missed_Payments': 'Missed Payments'
    },
    color_discrete_map={"Recovered": "green", "Not Recovered": "red"},
    opacity=0.8
)

# Graph layout clean karna
fig_3d_recovery.update_layout(
    margin=dict(l=0, r=0, b=0, t=40),
    legend_title="Recovery Status"
)
fig_3d_recovery.show()


### Case Study Part 11: Mapping Segment Names & Creating Risk Flags
We assign meaningful business names to each numerical cluster segment. Additionally, we define a binary classification target variable (`High_Risk_Flag`) representing high-risk segments.

In [11]:
# 1. Segment unique numbers ko meaningful naam dena
df['Segment_Name'] = df['Borrower_Segment'].map({
    0: 'Moderate Income, High Loan Burden',
    1: 'High Income, Low Default Risk',
    2: 'Moderate Income, Medium Risk',
    3: 'High Loan, Higher Default Risk'
})

# 2. Target Variable 'High_Risk_Flag' (0 ya 1) banana Random Forest ke liye
df['High_Risk_Flag'] = df['Segment_Name'].apply(lambda x: 1 if x in ['High Loan, Higher Default Risk', 'Moderate Income, High Loan Burden'] else 0)

print("🏷️ Labels mapping aur Target definitions set ho gayi hain!")
print("Target counts:\n", df['High_Risk_Flag'].value_counts())


🏷️ Labels mapping aur Target definitions set ho gayi hain!
Target counts:
 High_Risk_Flag
0    324
1    176
Name: count, dtype: int64


### Case Study Part 12: Supervised Risk Classification Model
Using the mapped features, we split the data into training and test sets and train a Random Forest Classifier. The model predicts the probability of a customer falling into a high-risk category, achieving strong evaluation scores.

In [12]:
X = df[features]
y = df['High_Risk_Flag']

# Train-Test Split (80% training, 20% testing with stratify to keep balance)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Hyperparameter tuned Random Forest model to lock 90%+ Accuracy
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12,
    min_samples_split=4,
    class_weight='balanced', # Data balance karne ke liye
    random_state=42
)
rf_model.fit(X_train, y_train)

# Predictions nikalna evaluation ke liye
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("="*50)
print(f"🔥 BOOM! Model Accuracy Achieved: {accuracy * 100:.2f}%")
print("="*50)
print("\n📝 Detailed Classification Report:")
print(classification_report(y_test, y_pred))


🔥 BOOM! Model Accuracy Achieved: 88.00%

📝 Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.91      0.91        65
           1       0.83      0.83      0.83        35

    accuracy                           0.88       100
   macro avg       0.87      0.87      0.87       100
weighted avg       0.88      0.88      0.88       100



### Case Study Part 13: Risk-Based Decision and Recovery Policy Engine
Using the model's predicted risk probabilities, we establish a rule-based policy engine. Borrowers are automatically segmented into low, medium, or high-risk categories, each tied to a specific action plan.

In [13]:
# Test data par Risk Scores (probabilites) nikalna
risk_scores = rf_model.predict_proba(X_test)[:, 1]

df_test = X_test.copy()
df_test['Risk_Score'] = risk_scores
df_test['Predicted_High_Risk'] = (df_test['Risk_Score'] > 0.5).astype(int)

# Original details merge karna safely
merge_cols = ['Segment_Name', 'Recovery_Status', 'Collection_Method', 'Collection_Attempts', 'Legal_Action_Taken']
if 'Borrower_ID' in df.columns:
    merge_cols.append('Borrower_ID')

df_test = df_test.merge(df[merge_cols], left_index=True, right_index=True)

# Policy Engine Engine logic function
def assign_recovery_strategy(risk_score):
    if risk_score > 0.75:
        return "Immediate legal notices & aggressive recovery attempts"
    elif 0.50 <= risk_score <= 0.75:
        return "Settlement offers & repayment plans"
    else:
        return "Automated reminders & monitoring"

df_test['Recovery_Strategy'] = df_test['Risk_Score'].apply(assign_recovery_strategy)

print("📊 Final Output Pipeline Preview (First 5 Rows):")
df_test[['Segment_Name', 'Risk_Score', 'Recovery_Strategy']].head()


📊 Final Output Pipeline Preview (First 5 Rows):


,Segment_Name,Risk_Score,Recovery_Strategy
197,"High Income, Low Default Risk",0.013333,Automated reminders & monitoring
413,"Moderate Income, Medium Risk",0.005240,Automated reminders & monitoring
489,"Moderate Income, Medium Risk",0.010883,Automated reminders & monitoring
259,"High Income, Low Default Risk",0.437339,Automated reminders & monitoring
435,"Moderate Income, High Loan Burden",0.993333,Immediate legal notices & aggressive recovery ...


### Case Study Part 14: Model Serialization and Export
To make our analytics pipeline reusable in production, we serialize and save both the trained Random Forest model and the fitted StandardScaler using `joblib`.

In [14]:
import joblib

# 1. Trained Random Forest Model ko save karna
joblib.dump(rf_model, 'loan_rf_model.joblib')

# 2. Preprocessing Scaler ko save karna
joblib.dump(scaler, 'loan_scaler.joblib')

print("💾 Success! Model ('loan_rf_model.joblib') aur Scaler ('loan_scaler.joblib') successfully save ho gaye hain.")


💾 Success! Model ('loan_rf_model.joblib') aur Scaler ('loan_scaler.joblib') successfully save ho gaye hain.


### Case Study Part 15: Scoring System Testing (Production Simulation)
We simulate a production endpoint by loading our saved model and scaler. We then pass an unseen dummy customer profile through the pipeline to output a live risk percentage and an automated recovery strategy.

In [15]:
import numpy as np
import pandas as pd
import joblib

# 1. Saved Model aur Scaler ko wapas load karna
loaded_rf_model = joblib.load('loan_rf_model.joblib')
loaded_loan_scaler = joblib.load('loan_scaler.joblib')

# 2. Naye Customer ka Dummy Data (Features strictly mapping the original list)
# Features order: ['Age', 'Monthly_Income', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate',
#                  'Collateral_Value', 'Outstanding_Loan_Amount', 'Monthly_EMI', 'Num_Missed_Payments', 'Days_Past_Due']
features_list = ['Age', 'Monthly_Income', 'Loan_Amount', 'Loan_Tenure', 'Interest_Rate',
                 'Collateral_Value', 'Outstanding_Loan_Amount', 'Monthly_EMI', 'Num_Missed_Payments', 'Days_Past_Due']

# Sample Data: Age=45, Income=8500, Loan=15000, Tenure=60, Rate=10.5, Collateral=20000, Outstanding=12000, EMI=350, Missed=4, DPD=45
new_borrower_data = [[45, 8500, 15000, 60, 10.5, 20000, 12000, 350, 4, 45]]

# 3. Warning se bachne ke liye proper Pandas DataFrame banana
new_borrower_df = pd.DataFrame(new_borrower_data, columns=features_list)

# 4. Predict Proba se exact Risk Score (Probability) nikalna
risk_percentage = loaded_rf_model.predict_proba(new_borrower_df)[:, 1][0]
predicted_class = loaded_rf_model.predict(new_borrower_df)[0]

# 5. Dynamic Recovery Strategy Assignment Logic
def check_strategy(score):
    if score > 0.75:
        return "Immediate legal notices & aggressive recovery attempts 🔴"
    elif 0.50 <= score <= 0.75:
        return "Settlement offers & repayment plans 🟡"
    else:
        return "Automated reminders & monitoring 🟢"

final_strategy = check_strategy(risk_percentage)

print("="*60)
print(f"📊 New Customer Analysis Results:")
print(f"📈 Calculated Risk Score: {risk_percentage * 100:.2f}%")
print(f"🎯 High Risk Classification Flag: {predicted_class}")
print(f"🛠️ Assigned Action Strategy: {final_strategy}")
print("="*60)


📊 New Customer Analysis Results:
📈 Calculated Risk Score: 1.19%
🎯 High Risk Classification Flag: 0
🛠️ Assigned Action Strategy: Automated reminders & monitoring 🟢


### Case Study Part 16: Feature Importance Interpretation
To make our model explainable, we extract feature importances from the Random Forest model and display them on an interactive horizontal bar chart. This tells business stakeholders which indicators are most predictive of loan defaults.

In [16]:
import pandas as pd
import plotly.express as px

# 1. Random Forest Model se Feature Importances nikalna
importances = rf_model.feature_importances_

# 2. Features aur unke importance scores ka DataFrame banana
feature_imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=True) # Ascending order taaki graph mein top par sabse bada aaye

# 3. Plotly se Horizontal Bar Graph banana
fig_imp = px.bar(
    feature_imp_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title="System Features Impact (Random Forest Feature Importance)",
    labels={'Importance': 'Impact Score (Weight)', 'Feature': 'System Features'},
    color='Importance',
    color_continuous_scale='Viridis'
)

# Graph ka layout clean aur responsive banana
fig_imp.update_layout(
    template="plotly_white",
    xaxis_title="Impact Score / Importance",
    yaxis_title="System Features",
    coloraxis_showscale=False,
    height=500
)

# Graph ko screen par display karna
fig_imp.show()
